# BirdCLEF 2026 Inference v15 — BCE + SpecAugment, 3-crop TTA
## Paired with training v15 | 10 models (5 folds x 2 archs) | 3-crop TTA
## Speed: batched per-soundscape, parallel CPU mel, torch.inference_mode


In [1]:
# === CELL 1: IMPORTS & CONFIG (must match v13 training exactly) ===
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)

import os, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast

import timm
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── CONFIG — must match v13 training exactly ─────────────────────────────────
CFG = dict(
    sr            = 16000,
    seconds       = 5,
    n_mels        = 64,
    n_fft         = 1024,
    hop           = 320,
    fmin          = 60,
    fmax          = 8000,           # v13: sr//2 = 8000, matching v7's proven default
    architectures = ["resnet18", "efficientnet_b0"],  # v13: v7's exact pair
    folds         = 5,              # v13: 5 folds -> 10 models total
    tta_offsets   = [-1.25, 0.0, 1.25],  # v15: 3-crop TTA (+/- 1.25s from center)
    device        = "cuda" if torch.cuda.is_available() else "cpu",
)

device = torch.device(CFG["device"])

# Use all available CPU cores for PyTorch ops (critical on CPU-only Kaggle)
torch.set_num_threads(os.cpu_count() or 4)
print("v15 inference config ready")
print(f"   Device     : {device}")
print(f"   Models     : {len(CFG['architectures'])} archs x {CFG['folds']} folds = {len(CFG['architectures'])*CFG['folds']}")
print(f"   TTA crops  : {len(CFG['tta_offsets'])}")
print(f"   fmax       : {CFG['fmax']} Hz")


v14 inference config ready
   Device     : cpu
   Models     : 2 archs x 5 folds = 10
   TTA crops  : 1
   fmax       : 8000 Hz


In [2]:
# === CELL 2: DATA PATHS & SPECIES ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV  = _first_existing(
    "/kaggle/input/birdclef-2026/taxonomy.csv",
    "/kaggle/input/competitions/birdclef-2026/taxonomy.csv",
)
TEST_AUDIO    = _first_existing(
    "/kaggle/input/birdclef-2026/test_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/test_soundscapes",
)
SAMPLE_SUB    = _first_existing(
    "/kaggle/input/birdclef-2026/sample_submission.csv",
    "/kaggle/input/competitions/birdclef-2026/sample_submission.csv",
)
# Checkpoint dataset (upload after training, or use /kaggle/working/ if same session)
CKPT_DIR      = _first_existing(
    "/kaggle/input/birdclef-2026-v15-model",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species",
    "/kaggle/working",
)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df["primary_label"].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

print(f"✅ {n_classes} species loaded from taxonomy.csv")
print(f"   TEST_AUDIO : {TEST_AUDIO}")
print(f"   CKPT_DIR   : {CKPT_DIR}")

✅ 234 species loaded from taxonomy.csv
   TEST_AUDIO : /kaggle/input/competitions/birdclef-2026/test_soundscapes
   CKPT_DIR   : /kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species


In [3]:
# === CELL 3: MEL HELPER (identical params to v9 training) ===
def logmel_from_wave(wave: np.ndarray, sr: int) -> np.ndarray:
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr,
        n_fft=CFG["n_fft"],
        hop_length=CFG["hop"],
        n_mels=CFG["n_mels"],
        fmin=CFG["fmin"],
        fmax=CFG["fmax"],
        power=2.0,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min, S_max = S_db.min(), S_db.max()
    if S_max - S_min < 1e-9:
        return np.zeros_like(S_db, dtype=np.float32)
    return np.clip((S_db - S_min) / (S_max - S_min + 1e-9), 0.0, 1.0).astype(np.float32)

print("✅ logmel_from_wave defined")
print(f"   n_mels={CFG['n_mels']}, n_fft={CFG['n_fft']}, fmax={CFG['fmax']}Hz")

✅ logmel_from_wave defined
   n_mels=64, n_fft=1024, fmax=8000Hz


In [4]:
# === CELL 4: MODEL ARCHITECTURE (pretrained=False for inference) ===
class BirdCLEFModel(nn.Module):
    def __init__(self, arch: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        self.arch = arch

        if arch == "resnet18":  # v13 primary backbone
            self.backbone = timm.create_model("resnet18", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features

        elif arch == "resnet50":  # kept for compatibility
            self.backbone = timm.create_model("resnet50", pretrained=pretrained, num_classes=0)
            self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            in_features = self.backbone.num_features

        elif arch in ("efficientnet_b0", "efficientnet_b2"):
            self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)
            stem = self.backbone.conv_stem
            self.backbone.conv_stem = nn.Conv2d(
                1, stem.out_channels,
                kernel_size=stem.kernel_size, stride=stem.stride,
                padding=stem.padding, bias=False,
            )
            in_features = self.backbone.num_features

        else:
            raise ValueError(f"Unsupported arch: {arch}")

        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

print("✅ BirdCLEFModel defined")

✅ BirdCLEFModel defined


In [5]:
# === CELL 5: LOAD 9 CHECKPOINTS (3 folds × 3 architectures) ===
models = []
missing = []

for arch in CFG["architectures"]:
    for fold_idx in range(CFG["folds"]):
        ckpt_path = Path(CKPT_DIR) / f"{arch}_v15_fold{fold_idx}.pt"
        if not ckpt_path.exists():
            missing.append(str(ckpt_path))
            continue
        m = BirdCLEFModel(arch, n_classes=n_classes, pretrained=False).to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
        m.eval()
        models.append(m)
        print(f"   ✅ Loaded {ckpt_path.name}")

if missing:
    print(f"\n⚠️  {len(missing)} checkpoint(s) not found:")
    for p in missing:
        print(f"      {p}")

print(f"\n✅ {len(models)}/{len(CFG['architectures'])*CFG['folds']} models loaded")

   ✅ Loaded resnet18_v14_fold0.pt
   ✅ Loaded resnet18_v14_fold1.pt
   ✅ Loaded resnet18_v14_fold2.pt
   ✅ Loaded resnet18_v14_fold3.pt
   ✅ Loaded resnet18_v14_fold4.pt
   ✅ Loaded efficientnet_b0_v14_fold0.pt
   ✅ Loaded efficientnet_b0_v14_fold1.pt
   ✅ Loaded efficientnet_b0_v14_fold2.pt
   ✅ Loaded efficientnet_b0_v14_fold3.pt
   ✅ Loaded efficientnet_b0_v14_fold4.pt

✅ 10/10 models loaded


In [6]:
# === CELL 6: BATCHED PREDICTION PER SOUNDSCAPE ===
_use_amp    = (device.type == "cuda")
win_samples = CFG["sr"] * CFG["seconds"]
win_frames  = int(CFG["seconds"] * CFG["sr"] / CFG["hop"]) + 1
_n_tta      = len(CFG["tta_offsets"])
_pad_samps  = win_samples + int(max(abs(o) for o in CFG["tta_offsets"]) * CFG["sr"]) + 1
_CHUNK      = 512   # max mel-crops per GPU forward-pass (OOM safety)
# On CPU-only, leave cores for PyTorch's internal thread pool
_N_WORKERS  = (2 if device.type == "cpu" else min(4, (os.cpu_count() or 1)))


def _extract_mel_crop(args):
    """Extract and compute one mel crop from a pre-loaded padded waveform (thread-safe)."""
    wave_padded, nom_start, offset_s = args
    off   = int(offset_s * CFG["sr"])
    start = max(0, nom_start + off)
    end_  = start + win_samples
    if end_ > len(wave_padded):
        end_  = len(wave_padded)
        start = max(0, end_ - win_samples)
    clip = wave_padded[start:end_]
    if len(clip) < win_samples:
        clip = np.pad(clip, (0, win_samples - len(clip)))
    mel = logmel_from_wave(clip, CFG["sr"])
    T = mel.shape[1]
    if T < win_frames:
        mel = np.concatenate(
            [mel, np.zeros((mel.shape[0], win_frames - T), dtype=np.float32)], axis=1
        )
    elif T > win_frames:
        mel = mel[:, :win_frames]
    return mel


def predict_soundscape(audio_path: str, end_seconds) -> np.ndarray:
    """
    Load audio ONCE per soundscape.
    Compute all (n_rows x n_tta) mel crops in PARALLEL on CPU threads.
    Run all models in a SINGLE batched forward pass (GPU if available, else CPU).
    Returns ndarray (len(end_seconds), n_classes).
    """
    end_seconds = list(end_seconds)
    n_rows = len(end_seconds)

    if not models:
        return np.full((n_rows, n_classes), 0.5, dtype=np.float32)

    try:
        wave, sr0 = sf.read(audio_path, always_2d=False)
    except Exception:
        return np.full((n_rows, n_classes), 0.5, dtype=np.float32)

    if sr0 != CFG["sr"]:
        wave = librosa.resample(wave, orig_sr=sr0, target_sr=CFG["sr"])
    if wave.ndim == 2:
        wave = wave.mean(axis=1)
    wave = np.pad(wave.astype(np.float32), (_pad_samps, _pad_samps))

    # Build task list: one entry per (row x TTA) crop
    tasks = []
    for end_s in end_seconds:
        nom_end   = int(end_s * CFG["sr"]) + _pad_samps
        nom_start = nom_end - win_samples
        for offset_s in CFG["tta_offsets"]:
            tasks.append((wave, nom_start, offset_s))

    # Parallel mel computation across CPU threads
    with ThreadPoolExecutor(max_workers=_N_WORKERS) as pool:
        mels = list(pool.map(_extract_mel_crop, tasks))

    # (n_rows * n_tta, 1, n_mels, win_frames) on GPU
    batch = torch.from_numpy(np.stack(mels)[:, None]).float().to(device)

    all_model_probs = []
    for m in models:
        chunks = []
        for i in range(0, len(batch), _CHUNK):
            with torch.inference_mode(), autocast(enabled=_use_amp):
                p = torch.sigmoid(m(batch[i : i + _CHUNK])).float().cpu().numpy()
            chunks.append(p)
        probs = np.concatenate(chunks, axis=0)                            # (n_rows*n_tta, n_classes)
        probs = probs.reshape(n_rows, _n_tta, n_classes).mean(axis=1)    # (n_rows, n_classes)
        all_model_probs.append(probs)

    return np.mean(all_model_probs, axis=0).astype(np.float32)            # (n_rows, n_classes)


status = f"{len(models)} models" if models else "WARNING: 0 models loaded — will output neutral 0.5"
print("predict_soundscape() defined — parallel CPU mel + batched inference")
print(f"   CPU mel workers : {_N_WORKERS}")
print(f"   TTA crops       : {_n_tta}  offsets = {CFG['tta_offsets']} s")
print(f"   GPU chunk       : {_CHUNK} mel crops per forward pass")
print(f"   Models          : {status}")


predict_soundscape() defined — parallel CPU mel + batched inference
   CPU mel workers : 2
   TTA crops       : 1  offsets = [0.0] s
   GPU chunk       : 512 mel crops per forward pass
   Models          : 10 models


In [7]:
# === CELL 7: GENERATE PREDICTIONS ===
sample_sub = pd.read_csv(SAMPLE_SUB)
print(f"Sample submission rows: {len(sample_sub)}")
print(f"Columns: {list(sample_sub.columns[:5])} ...")

# Accumulate as numpy arrays — avoids building 50k+ dicts and slow pd.DataFrame(list_of_dicts)
all_row_ids    = []
all_probs_list = []    # list of (n_rows, n_classes) float32 arrays
missing_audio  = 0

sample_sub = sample_sub.copy()
sample_sub["_soundscape_id"] = sample_sub["row_id"].str.rsplit("_", n=1).str[0]

for soundscape_id, group in tqdm(
    sample_sub.groupby("_soundscape_id"),
    desc="Soundscapes",
    unit="file",
):
    audio_path = None
    for ext in [".ogg", ".wav", ".flac"]:
        cand = Path(TEST_AUDIO) / f"{soundscape_id}{ext}"
        if cand.exists():
            audio_path = str(cand)
            break

    row_ids = [str(r) for r in group["row_id"]]

    if audio_path is None:
        missing_audio += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    end_seconds = [int(rid.rsplit("_", 1)[-1]) for rid in row_ids]
    probs_all   = predict_soundscape(audio_path, end_seconds)  # (n_rows, n_classes)

    all_row_ids.extend(row_ids)
    all_probs_list.append(probs_all)

if missing_audio:
    print(f"⚠️  {missing_audio} soundscape(s) not found in {TEST_AUDIO} — used neutral 0.5")
print(f"\n✅ Generated {len(all_row_ids)} prediction rows")


Sample submission rows: 3
Columns: ['row_id', '1161364', '116570', '1176823', '1491113'] ...


Soundscapes: 100%|██████████| 1/1 [00:00<00:00, 163.28file/s]

⚠️  1 soundscape(s) not found in /kaggle/input/competitions/birdclef-2026/test_soundscapes — used neutral 0.5

✅ Generated 3 prediction rows


In [8]:
# === CELL 8: BUILD SUBMISSION ===
submission_df = sample_sub[["row_id"]].copy()
submission_df["row_id"] = submission_df["row_id"].astype(str)

if all_probs_list:
    # Stack into (N, n_classes) and build DataFrame in one shot — much faster than list-of-dicts
    probs_np = np.vstack(all_probs_list).astype(np.float64)   # (N, n_classes)
    pred_df  = pd.DataFrame(probs_np, columns=species)
    pred_df.insert(0, "row_id", all_row_ids)
    pred_df["row_id"] = pred_df["row_id"].astype(str)
    submission_df = submission_df.merge(pred_df, on="row_id", how="left")
else:
    print("⚠️  No predictions — building fully neutral submission (0.5)")

# Fill any missing species columns (left-join NaN or fully absent)
for sp in species:
    if sp in submission_df.columns:
        submission_df[sp] = submission_df[sp].fillna(0.5).astype(np.float64)
    else:
        submission_df[sp] = 0.5

submission_df.to_csv("/kaggle/working/submission.csv", index=False)
print(f"✅ Saved submission.csv  ({len(submission_df)} rows × {len(submission_df.columns)} columns)")
submission_df.head(3)


✅ Saved submission.csv  (3 rows × 235 columns)


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,BC2026_Test_0001_S05_20250227_010002_10,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
2,BC2026_Test_0001_S05_20250227_010002_15,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [9]:
# === CELL 9: FORMAT VALIDATION ===
print("Running submission validation...")
issues = []

# Check row_id dtype
if submission_df["row_id"].dtype != object:
    issues.append(f"row_id dtype is {submission_df['row_id'].dtype}, expected str")

# Check species columns exist and are numeric
missing_cols = [sp for sp in species if sp not in submission_df.columns]
if missing_cols:
    issues.append(f"{len(missing_cols)} species columns missing: {missing_cols[:5]}")

# Check value range
sp_cols  = [c for c in submission_df.columns if c != "row_id"]
val_min  = submission_df[sp_cols].min().min()
val_max  = submission_df[sp_cols].max().max()
if val_min < 0 or val_max > 1:
    issues.append(f"Predictions out of [0,1]: min={val_min:.4f} max={val_max:.4f}")

# Check for NaN
nan_count = submission_df[sp_cols].isna().sum().sum()
if nan_count > 0:
    issues.append(f"{nan_count} NaN values in species columns")

if issues:
    print("\n⚠️  Issues found:")
    for issue in issues:
        print(f"   • {issue}")
else:
    print("✅ Submission looks valid!")
    print(f"   Rows        : {len(submission_df)}")
    print(f"   Species cols: {len(sp_cols)}")
    print(f"   Value range : [{val_min:.4f}, {val_max:.4f}]")
    print(f"   NaN count   : {nan_count}")

Running submission validation...
✅ Submission looks valid!
   Rows        : 3
   Species cols: 234
   Value range : [0.5000, 0.5000]
   NaN count   : 0
